# Week 7: Evaluate your fine-tuned pricer

Load the test set, your fine-tuned model (base + PEFT adapter), and run the official evaluation. The reported metric is average $ error on the test set; the instructor's reference is 39.85 (Fine-tuned Full).

## 1. Setup and path (Colab or local)

Run from repo root or from `week 7 exercise`; we add `week7` to the path so `pricer` resolves.

In [ ]:
import os
import sys
import re
from pathlib import Path

# Add week7 so pricer is found (Colab: clone repo into content; local: run from repo root or week 7 exercise)
_cwd = Path.cwd()
if (_cwd / "week7" / "pricer").exists():
    sys.path.insert(0, str(_cwd / "week7"))
elif (_cwd / "pricer").exists():
    sys.path.insert(0, str(_cwd))
else:
    sys.path.insert(0, str(_cwd.parent.parent.parent))  # from week 7 exercise folder

from dotenv import load_dotenv
load_dotenv(override=True)

# Colab: use userdata.get('HF_TOKEN'); local: use env
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN')

from huggingface_hub import login
login(HF_TOKEN, add_to_git_credential=True)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, set_seed
from peft import PeftModel
from tqdm.auto import tqdm

from pricer.items import Item
from pricer.evaluator import evaluate

## 2. Config: data, base model, and your fine-tuned run

Set `RUN_NAME` and `REVISION` to the run and checkpoint you want to evaluate (e.g. your best training run).

In [ ]:
LITE_MODE = False
username = "Olasheni"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

BASE_MODEL = "meta-llama/Llama-3.2-3B"
PROJECT_NAME = "price"
HF_USER = "Olasheni"

RUN_NAME = "2026-03-13_10.31.10-lite"
REVISION = None  # Lite run: use default branch; for a specific checkpoint use the commit hash
FINETUNED_MODEL = f"{HF_USER}/{PROJECT_NAME}-{RUN_NAME}"

QUANT_4_BIT = True  # 4-bit for T4; instructor used 4-bit
EVAL_SIZE = 200    # test set size for evaluation (same as evaluator default)

In [ ]:
train, val, test = Item.from_hub(dataset)
print(f"Train: {len(train):,}, Val: {len(val):,}, Test: {len(test):,}")

## 3. Load base model + PEFT adapter

Base model: 4-bit quantized Llama 3.2 3B. Then load LoRA adapters (instructor's or yours) with the chosen revision.

In [ ]:
if QUANT_4_BIT:
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_type="nf4"
    )
else:
    quant_config = BitsAndBytesConfig(
        load_in_8bit=True,
        bnb_8bit_compute_dtype=torch.bfloat16
    )

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
)
base_model.generation_config.pad_token_id = tokenizer.pad_token_id

fine_tuned_model = PeftModel.from_pretrained(base_model, FINETUNED_MODEL, revision=REVISION)
fine_tuned_model.eval()
print(f"Model loaded. Memory footprint: {fine_tuned_model.get_memory_footprint() / 1e9:.2f} GB")

## 4. Predictor and evaluation

Predictor uses the same prompt format as training (`item.test_prompt()`). Evaluator reports average $ error and charts.

In [ ]:
from pricer.items import PREFIX

def model_predict(item):
    set_seed(42)
    prompt = item.test_prompt()
    inputs = tokenizer(prompt, return_tensors="pt").to(fine_tuned_model.device)
    with torch.no_grad():
        out = fine_tuned_model.generate(
            **inputs,
            max_new_tokens=16,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    full = tokenizer.decode(out[0], skip_special_tokens=True)
    if PREFIX in full:
        content = full.split(PREFIX)[1].strip()
        content = content.replace(",", "")
        match = re.search(r"[-+]?\d*\.\d+|\d+", content)
        return float(match.group()) if match else 0.0
    return 0.0

# Wrapper so evaluator gets a callable that takes item and returns value (post_process handles it)
def predictor(item):
    return model_predict(item)

In [ ]:
evaluate(predictor, test, size=EVAL_SIZE)

## 5. Next run

Set `RUN_NAME` and `REVISION` in section 2 to another checkpoint and re-run to compare.